# CSE — Anomaly Detection Baseline

Detect unusual trading activity across CSE equities.

**Covers:**
1. Rule-based anomalies (price spikes, volume spikes, OHLC violations)
2. Isolation Forest on per-stock feature vectors
3. Visualisation of flagged anomalous days

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
df = pd.read_parquet('../data/published/cse_unified.parquet')
df['date'] = pd.to_datetime(df['date'])

# Focus on trading days only
df = df[df['is_trading_day']].copy().reset_index(drop=True)
print(f'Trading rows: {len(df)}  |  Symbols: {df["symbol"].nunique()}')

## 1. Rule-Based Anomaly Flags

In [ ]:
# Price spike: |return_1d| > 20%
df['flag_price_spike'] = df['return_1d'].abs() > 0.20

# Volume spike: volume_zscore > 4
df['flag_volume_spike'] = df['volume_zscore'].fillna(0) > 4.0

# OHLC violation already present
df['flag_ohlc'] = df['ohlc_invalid'].astype(bool)

# Combined rule flag
df['flag_any_rule'] = df['flag_price_spike'] | df['flag_volume_spike'] | df['flag_ohlc']

print('Rule-based anomaly counts:')
print(f'  Price spike (|ret| > 20%):   {df["flag_price_spike"].sum():,}')
print(f'  Volume spike (zscore > 4):   {df["flag_volume_spike"].sum():,}')
print(f'  OHLC invalid:                {df["flag_ohlc"].sum():,}')
print(f'  Any rule flag:               {df["flag_any_rule"].sum():,} ({100*df["flag_any_rule"].mean():.2f}%)')

## 2. Rule Anomaly Distribution Over Time

In [ ]:
daily_anomalies = df.groupby('date')['flag_any_rule'].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_anomalies.index, daily_anomalies.values, lw=0.8, color='crimson')
ax.set_ylabel('Anomalous stocks per day')
ax.set_title('Daily Count of Rule-Based Anomalies Across All Stocks')
plt.tight_layout()
plt.show()

# Top 10 anomaly dates
print('Top 10 dates with most anomalous stocks:')
print(daily_anomalies.sort_values(ascending=False).head(10).to_string())

## 3. Isolation Forest — Per-Row Anomaly Score

In [ ]:
FEAT_COLS = ['return_1d', 'return_5d', 'volatility_20d',
             'volume_zscore', 'close_to_ma50', 'close_to_ma200']

model_df = df[FEAT_COLS + ['date', 'symbol']].dropna()
print(f'Rows for Isolation Forest: {len(model_df):,}')

X = model_df[FEAT_COLS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.02,
                       random_state=42, n_jobs=-1)
model_df = model_df.copy()
model_df['iso_score'] = iso.fit_predict(X_scaled)  # -1 = anomaly, 1 = normal
model_df['iso_anomaly'] = model_df['iso_score'] == -1
model_df['iso_raw'] = iso.score_samples(X_scaled)   # lower = more anomalous

print(f'Isolation Forest anomalies: {model_df["iso_anomaly"].sum():,} '
      f'({100*model_df["iso_anomaly"].mean():.2f}%)')

## 4. Anomaly Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(model_df['iso_raw'], bins=100, color='steelblue', edgecolor='none')
threshold = model_df[model_df['iso_anomaly']]['iso_raw'].max()
ax.axvline(threshold, color='red', lw=1.5, linestyle='--', label=f'Anomaly threshold ≈ {threshold:.3f}')
ax.set_xlabel('Isolation Forest anomaly score (lower = more anomalous)')
ax.set_ylabel('Count')
ax.set_title('Isolation Forest Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Top Anomalous Events

In [ ]:
top_anomalies = (
    model_df[model_df['iso_anomaly']]
    .nsmallest(20, 'iso_raw')[['date', 'symbol'] + FEAT_COLS + ['iso_raw']]
)
print('Top 20 most anomalous (date, symbol) observations:')
print(top_anomalies.to_string(index=False))

## 6. Visualise Anomalies for a Single Stock

In [ ]:
# Pick the stock with the most anomalies
most_anomalous_sym = model_df[model_df['iso_anomaly']]['symbol'].value_counts().idxmax()
print(f'Stock with most anomalies: {most_anomalous_sym}')

stock_df   = df[df['symbol'] == most_anomalous_sym].sort_values('date')
anom_dates = model_df[(model_df['iso_anomaly']) & (model_df['symbol'] == most_anomalous_sym)]['date']

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(stock_df['date'], stock_df['close'], lw=0.9, color='steelblue', label='Close')
for d in anom_dates:
    axes[0].axvline(d, color='red', alpha=0.4, lw=0.8)
axes[0].set_ylabel('Close price (LKR)')
axes[0].set_title(f'{most_anomalous_sym} — Close Price with Isolation Forest Anomalies (red)')

axes[1].bar(stock_df['date'], stock_df['volume'], width=1, color='steelblue', alpha=0.6)
for d in anom_dates:
    axes[1].axvline(d, color='red', alpha=0.4, lw=0.8)
axes[1].set_ylabel('Volume')
axes[1].set_title('Volume')

plt.tight_layout()
plt.show()

## 7. Agreement Between Rule-Based and Isolation Forest

In [ ]:
merged = model_df[['date','symbol','iso_anomaly']].merge(
    df[['date','symbol','flag_any_rule']], on=['date','symbol'], how='left'
)
both  = (merged['iso_anomaly'] & merged['flag_any_rule']).sum()
iso_only  = (merged['iso_anomaly'] & ~merged['flag_any_rule']).sum()
rule_only = (~merged['iso_anomaly'] & merged['flag_any_rule']).sum()

print('Agreement between rule-based and Isolation Forest anomalies:')
print(f'  Both flagged:          {both:,}')
print(f'  Isolation Forest only: {iso_only:,}')
print(f'  Rule-based only:       {rule_only:,}')